In [1]:
import torch 
import torch.nn as nn
import numpy as np
from pathlib import Path
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from src.data.tabular import split_data
from src.models.Autoencoder import Autoencoder

In [2]:
digits = load_digits()
X = digits.data
X = StandardScaler().fit_transform(X)  
y = digits.target
X_train, X_val, X_test, y_train, y_val, y_test, encoder = split_data(X, y)
X_train_tensor = torch.from_numpy(X_train).float()
X_val_tensor   = torch.from_numpy(X_val).float()
X_test_tensor  = torch.from_numpy(X_test).float()

X_train_val_tensor = torch.cat([X_train_tensor, X_val_tensor], dim=0)
X_train_val = X_train_val_tensor.detach().cpu().numpy()
print("split shape",X_train.shape, X_val.shape, X_test.shape)

n_features = X_train.shape[1]

split shape (1078, 64) (360, 64) (359, 64)


In [3]:
import torch
from src.hpo.autoencoder_hpo import create_study

X_train_tensor = torch.randn(8, 4)
X_val_tensor = torch.randn(4, 4)

study = create_study(
    X_train_tensor,
    X_val_tensor,
    n_epochs=2,
    bottleneck_dim=2,
    n_trials=20,
    study_name="autoencoder_HPO_digits_analysis"
)

print(len(study.trials))
print(study.best_value)
print(study.best_params)

h:\Course contents FAU\semester 5\Thesis\final Project\LatentDimAnalysis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-09-07 20:54:08,504] A new study created in memory with name: autoencoder_HPO_digits_analysis
Best trial: 6. Best value: 1.22101:  30%|███       | 6/20 [00:03<00:07,  1.97it/s]

[I 2026-09-07 20:54:11,736] Trial 0 finished with value: 1.4957855939865112 and parameters: {'lr': 0.0001329291894316216, 'depth': 3, 'activation': 'ReLU'}. Best is trial 0 with value: 1.4957855939865112.
[I 2026-09-07 20:54:11,773] Trial 1 finished with value: 1.6333657503128052 and parameters: {'lr': 1.493656855461762e-05, 'depth': 3, 'activation': 'Tanh'}. Best is trial 0 with value: 1.4957855939865112.
[I 2026-09-07 20:54:11,793] Trial 2 finished with value: 1.5447883605957031 and parameters: {'lr': 0.00314288089084011, 'depth': 1, 'activation': 'Tanh'}. Best is trial 0 with value: 1.4957855939865112.
[I 2026-09-07 20:54:11,813] Trial 3 finished with value: 1.5320466756820679 and parameters: {'lr': 0.00019762189340280086, 'depth': 1, 'activation': 'ReLU'}. Best is trial 0 with value: 1.4957855939865112.
[I 2026-09-07 20:54:11,854] Trial 4 finished with value: 1.4013020992279053 and parameters: {'lr': 0.00023345864076016249, 'depth': 3, 'activation': 'GELU'}. Best is trial 4 with va

Best trial: 6. Best value: 1.22101:  65%|██████▌   | 13/20 [00:03<00:01,  6.70it/s]

[I 2026-09-07 20:54:11,941] Trial 7 finished with value: 1.4712049961090088 and parameters: {'lr': 1.2681352169084594e-05, 'depth': 3, 'activation': 'LeakyReLU'}. Best is trial 6 with value: 1.221008062362671.
[I 2026-09-07 20:54:11,963] Trial 8 finished with value: 1.2604767084121704 and parameters: {'lr': 0.00043664735929796326, 'depth': 1, 'activation': 'ReLU'}. Best is trial 6 with value: 1.221008062362671.
[I 2026-09-07 20:54:11,990] Trial 9 pruned. 
[I 2026-09-07 20:54:12,025] Trial 10 finished with value: 1.3755489587783813 and parameters: {'lr': 0.007141749214397002, 'depth': 2, 'activation': 'LeakyReLU'}. Best is trial 6 with value: 1.221008062362671.
[I 2026-09-07 20:54:12,056] Trial 11 finished with value: 1.3123949766159058 and parameters: {'lr': 5.608877985158643e-05, 'depth': 2, 'activation': 'ReLU'}. Best is trial 6 with value: 1.221008062362671.
[I 2026-09-07 20:54:12,078] Trial 12 pruned. 
[I 2026-09-07 20:54:12,102] Trial 13 pruned. 


Best trial: 6. Best value: 1.22101: 100%|██████████| 20/20 [00:03<00:00,  5.35it/s]

[I 2026-09-07 20:54:12,128] Trial 14 finished with value: 1.3797075748443604 and parameters: {'lr': 6.159474590220515e-05, 'depth': 1, 'activation': 'ReLU'}. Best is trial 6 with value: 1.221008062362671.
[I 2026-09-07 20:54:12,151] Trial 15 pruned. 
[I 2026-09-07 20:54:12,172] Trial 16 pruned. 
[I 2026-09-07 20:54:12,196] Trial 17 finished with value: 1.3406394720077515 and parameters: {'lr': 9.449908464788399e-05, 'depth': 1, 'activation': 'LeakyReLU'}. Best is trial 6 with value: 1.221008062362671.
[I 2026-09-07 20:54:12,221] Trial 18 pruned. 
[I 2026-09-07 20:54:12,242] Trial 19 pruned. 
20
1.221008062362671
{'lr': 8.200518402245828e-05, 'depth': 1, 'activation': 'ReLU'}


In [5]:
from sklearn.decomposition import PCA
from src.hpo.hidden_dims import build_hidden_dims_geo, build_hidden_dims_static
from src.training.train import train_final_model

best_params = study.best_params
activations = {
    'ReLU': nn.ReLU,
    'LeakyReLU': nn.LeakyReLU,
    'GELU': nn.GELU,
    'Tanh': nn.Tanh
}

best_activation = activations.get(best_params.get('activation', 'ReLU'), nn.ReLU)
best_depth = best_params['depth']
best_lr = best_params['lr']

input_dim = X_train_val_tensor.shape[1]
bottleneck_range = [1, 2, 4, 8, 16, 32, 62]
epochs = 400
ae_errors = []
pca_errors = []

for k in bottleneck_range:

    #final_hidden_dims = build_hidden_dims_geo(input_dim, k, best_depth)
    final_hidden_dims = build_hidden_dims_static(input_dim, depth=best_depth)

    pca = PCA(n_components=k)
    X_reduced = pca.fit_transform(X_train_val)
    X_recon_pca = pca.inverse_transform(X_reduced)
    pca_errors.append(np.mean((X_train_val - X_recon_pca) ** 2))

    final_model = Autoencoder(
        n_features=input_dim,
        bottleneck_dim=k,
        non_linear=True,
        non_linear_function=best_activation,
        hidden_dims_list=final_hidden_dims
    )

    train_loss, test_loss = train_final_model(
        final_model=final_model,
        X_train_val_tensor=X_train_val_tensor,
        X_test_tensor=X_test_tensor, 
        best_params=best_params, 
        epochs=400
    )

    ae_errors.append(train_loss)
    print(f"k={k:>3} PCA={pca_errors[-1]:.4f}  Non Linear AE={ae_errors[-1]:.4f}")

k=  1 PCA=0.8499  Non Linear AE=0.8022
k=  2 PCA=0.7585  Non Linear AE=0.7117
k=  4 PCA=0.6153  Non Linear AE=0.5648
k=  8 PCA=0.4561  Non Linear AE=0.5274
k= 16 PCA=0.2587  Non Linear AE=0.4793
k= 32 PCA=0.0855  Non Linear AE=0.4234
k= 62 PCA=0.0000  Non Linear AE=0.3653
